# 01 — Diagnóstico minucioso da base de entrevistas

Este notebook é a **etapa zero** do novo projeto: ele não cria ainda o pré-processamento final. O objetivo é entender a base antes de decidir regras de segmentação, limpeza, idioma, tokenização, stopwords, embeddings, clusters e geração de insights.

Ele puxa diretamente do BigQuery as **duas tabelas** do dataset:

1. `kyra-pml-20261.kyra_base_2026_1.entrevistas_chunks_v2` — base segmentada / chunks / tokens.
2. A outra tabela do mesmo dataset — base de entrevistas completas. Se houver só duas tabelas no dataset, ela é escolhida automaticamente. Se houver mais, o notebook ranqueia candidatas e deixa claro qual foi escolhida.

Todos os outputs são salvos em:

```text
../data/diagnostics/
```

Os snapshots baixados do BigQuery são salvos em:

```text
../data/raw/bq_snapshots/
```

**Importante:** rode este notebook com `Kernel > Restart Kernel and Run All Cells`. Não reaproveite células antigas do notebook anterior.


## 1. Instalação mínima de dependências

A célula abaixo só instala pacotes se estiverem faltando. Em ambiente corporativo, se a instalação automática não for permitida, instale no terminal do ambiente e rode novamente.


In [1]:
import sys
import importlib.util
import subprocess

REQUIRED_PACKAGES = {
    "google.cloud.bigquery": "google-cloud-bigquery",
    "db_dtypes": "db-dtypes",
    "pyarrow": "pyarrow",
    "pandas": "pandas",
    "numpy": "numpy",
}

missing = []
for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append(package_name)

if missing:
    print("Instalando dependências ausentes:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Dependências principais OK.")


Dependências principais OK.


## 2. Imports, paths e configuração


In [2]:
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import ast
import json
import math
import os
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 250)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_rows", 120)

# ---------------------------------------------------------------------
# Raiz do projeto
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
DIAG_DIR = DATA_DIR / "diagnostics"
BQ_CACHE_DIR = RAW_DIR / "bq_snapshots"
PLOTS_DIR = DIAG_DIR / "plots"

for p in [DATA_DIR, RAW_DIR, PROCESSED_DIR, DIAG_DIR, BQ_CACHE_DIR, PLOTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DIAG_DIR    :", DIAG_DIR)
print("BQ_CACHE_DIR:", BQ_CACHE_DIR)
print("RUN_ID      :", RUN_ID)

# ---------------------------------------------------------------------
# BigQuery
# ---------------------------------------------------------------------
LOAD_FROM_BIGQUERY = True

BQ_PROJECT = "kyra-pml-20261"
BQ_DATASET = "kyra_base_2026_1"
BQ_TOKENS_TABLE = "kyra-pml-20261.kyra_base_2026_1.entrevistas_chunks_v2"

# Deixe None para autodetectar a tabela de entrevistas completas.
# Se quiser travar manualmente depois, preencha com o nome completo:
# BQ_FULL_INTERVIEWS_TABLE = "kyra-pml-20261.kyra_base_2026_1.NOME_DA_TABELA_COMPLETA"
BQ_FULL_INTERVIEWS_TABLE = None

# Use None para baixar a tabela inteira. Para teste rápido, coloque um inteiro, ex.: 5000.
BQ_LIMIT_ROWS = None

# Se True, usa snapshots locais quando já existirem.
USE_BQ_CACHE = True


PROJECT_ROOT: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2
DIAG_DIR    : /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/diagnostics
BQ_CACHE_DIR: /Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/raw/bq_snapshots
RUN_ID      : 20260519_234444


## 3. Funções utilitárias


In [3]:
def save_df(df: pd.DataFrame, filename: str, index: bool = False) -> Path:
    """Salva CSV em data/diagnostics e devolve o path."""
    path = DIAG_DIR / filename
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"Salvo: {path.relative_to(PROJECT_ROOT)} | shape={df.shape}")
    return path


def save_json(obj, filename: str) -> Path:
    path = DIAG_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    print(f"Salvo: {path.relative_to(PROJECT_ROOT)}")
    return path


def save_text(txt: str, filename: str) -> Path:
    path = DIAG_DIR / filename
    path.write_text(txt, encoding="utf-8")
    print(f"Salvo: {path.relative_to(PROJECT_ROOT)}")
    return path


def short_path(path) -> str:
    if path is None:
        return "None"
    try:
        return str(Path(path).resolve().relative_to(PROJECT_ROOT))
    except Exception:
        return str(path)


def normalize_bq_table_name(table_name: str, default_project=BQ_PROJECT, default_dataset=BQ_DATASET) -> str:
    table_name = str(table_name).strip().strip("`")
    parts = table_name.split(".")
    if len(parts) == 3:
        return table_name
    if len(parts) == 2:
        return f"{default_project}.{table_name}"
    if len(parts) == 1:
        return f"{default_project}.{default_dataset}.{table_name}"
    raise ValueError(f"Nome de tabela inválido: {table_name}")


def safe_filename_from_table(table_fq: str) -> str:
    return table_fq.replace(".", "__").replace("-", "_") + ".parquet"


def read_bq_table(client, table_fq: str, limit_rows=None, use_cache=True) -> pd.DataFrame:
    table_fq = normalize_bq_table_name(table_fq)
    cache_file = BQ_CACHE_DIR / safe_filename_from_table(table_fq)

    if use_cache and cache_file.exists():
        print(f"Lendo cache local: {cache_file.relative_to(PROJECT_ROOT)}")
        return pd.read_parquet(cache_file)

    limit_sql = "" if limit_rows is None else f" LIMIT {int(limit_rows)}"
    query = f"SELECT * FROM `{table_fq}`{limit_sql}"
    print("Rodando query:", query)
    df = client.query(query).to_dataframe()
    df.to_parquet(cache_file, index=False)
    print(f"Snapshot salvo em: {cache_file.relative_to(PROJECT_ROOT)} | shape={df.shape}")
    return df


def normalize_text_basic(x) -> str:
    if pd.isna(x):
        return ""
    return str(x).replace("\x00", " ").strip()


def strip_accents_lower(text: str) -> str:
    text = normalize_text_basic(text).lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"\s+", " ", text).strip()
    return text


def text_for_hash(text: str) -> str:
    text = strip_accents_lower(text)
    text = re.sub(r"\d+", "0", text)
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def simple_word_tokens(text: str) -> list[str]:
    text = strip_accents_lower(text)
    return re.findall(r"[a-zA-ZáéíóúàâêôãõçñüÁÉÍÓÚÀÂÊÔÃÕÇÑÜ]{2,}", text)


def parse_possible_list(value):
    """Converte strings de lista em lista. Se não for lista, devolve None."""
    if isinstance(value, list):
        return value
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, str):
        s = value.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple)):
                    return list(parsed)
            except Exception:
                return None
    return None


def col_score(name: str, positive_terms: list[str], negative_terms: list[str] | None = None) -> int:
    n = name.lower()
    score = 0
    for term in positive_terms:
        if term in n:
            score += 1
    for term in (negative_terms or []):
        if term in n:
            score -= 1
    return score


def guess_column(df: pd.DataFrame, kind: str):
    cols = list(df.columns)
    lower_map = {c.lower(): c for c in cols}

    candidates_by_kind = {
        "doc_id": ["doc_id", "document_id", "entrevista_id", "id_entrevista", "id_documento", "arquivo_id", "file_id", "transcript_id", "id"],
        "chunk_id": ["chunk_id", "id_chunk", "segment_id", "trecho_id", "id_trecho"],
        "chunk_index": ["chunk_index", "chunk_idx", "idx", "ordem", "indice", "index", "posicao", "segment_index"],
        "language": ["idioma", "language", "lang", "lingua"],
        "text": ["texto_analise", "texto", "text", "transcricao", "transcrição", "conteudo", "content", "entrevista", "resposta", "fala", "chunk_text", "full_text", "texto_original", "texto_anonimizado"],
        "tokens": ["tokens", "token", "lista_tokens", "palavras", "words", "stems"],
        "date": ["data", "date", "created", "created_at", "timestamp", "dt"],
    }

    exacts = candidates_by_kind.get(kind, [])
    for cand in exacts:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]

    # Score por nome + tipo/estatística simples para texto.
    scored = []
    for c in cols:
        n = c.lower()
        score = col_score(n, exacts)
        if kind == "text":
            score += col_score(n, ["texto", "text", "trans", "conteudo", "content", "fala", "resposta"], ["id", "index", "idx", "token", "idioma", "language"])
            if df[c].dtype == object:
                sample = df[c].dropna().astype(str).head(200)
                if len(sample):
                    avg_len = sample.str.len().mean()
                    if avg_len > 80:
                        score += 3
                    elif avg_len > 30:
                        score += 1
        elif kind == "tokens":
            score += col_score(n, ["token", "stem", "lemma", "palavra", "word"], ["n_token", "count"])
            if df[c].dtype == object:
                sample = df[c].dropna().head(100)
                list_hits = sum(parse_possible_list(x) is not None for x in sample)
                if list_hits >= max(3, len(sample) * 0.20):
                    score += 5
        else:
            score += col_score(n, exacts)
        if score > 0:
            scored.append((score, c))

    if scored:
        return sorted(scored, reverse=True)[0][1]
    return None


def guess_columns(df: pd.DataFrame) -> dict:
    kinds = ["doc_id", "chunk_id", "chunk_index", "language", "text", "tokens", "date"]
    return {kind: guess_column(df, kind) for kind in kinds}


def dataframe_profile(df: pd.DataFrame, name: str) -> pd.DataFrame:
    rows = []
    for c in df.columns:
        s = df[c]
        non_null = int(s.notna().sum())
        n_unique = int(s.nunique(dropna=True)) if non_null else 0
        sample_values = s.dropna().astype(str).head(3).tolist()
        rows.append({
            "base": name,
            "coluna": c,
            "dtype": str(s.dtype),
            "n_linhas": len(df),
            "n_na": int(s.isna().sum()),
            "pct_na": round(float(s.isna().mean()), 4),
            "n_unique": n_unique,
            "pct_unique_sobre_nao_nulo": round(n_unique / non_null, 4) if non_null else None,
            "amostras": " | ".join(v[:160].replace("\n", " ") for v in sample_values),
        })
    return pd.DataFrame(rows)


## 4. BigQuery — catálogo e download das duas tabelas

Esta seção resolve o ponto crítico: baixar a tabela de chunks/tokens **e também** a tabela de entrevistas completas.


In [4]:
from google.cloud import bigquery

client = bigquery.Client(project=BQ_PROJECT)

# Catálogo do dataset
catalog_rows = []
for item in client.list_tables(f"{BQ_PROJECT}.{BQ_DATASET}"):
    table_fq = f"{item.project}.{item.dataset_id}.{item.table_id}"
    try:
        tbl = client.get_table(table_fq)
        schema = [{"name": f.name, "field_type": f.field_type, "mode": f.mode} for f in tbl.schema]
        schema_names = [f["name"] for f in schema]
        catalog_rows.append({
            "table_id": item.table_id,
            "table_fq": table_fq,
            "num_rows": tbl.num_rows,
            "num_bytes": tbl.num_bytes,
            "created": tbl.created,
            "modified": tbl.modified,
            "schema_names": schema_names,
            "schema": schema,
        })
    except Exception as e:
        catalog_rows.append({
            "table_id": item.table_id,
            "table_fq": table_fq,
            "error": repr(e),
        })

catalog = pd.DataFrame(catalog_rows)
if catalog.empty:
    raise RuntimeError(f"Nenhuma tabela encontrada em {BQ_PROJECT}.{BQ_DATASET}.")

save_df(
    catalog[[c for c in ["table_id", "table_fq", "num_rows", "num_bytes", "created", "modified", "schema_names", "error"] if c in catalog.columns]],
    "00_bq_catalogo_tabelas.csv",
)
display(catalog[[c for c in ["table_id", "table_fq", "num_rows", "num_bytes", "modified", "schema_names"] if c in catalog.columns]])

BQ_TOKENS_TABLE_RESOLVED = normalize_bq_table_name(BQ_TOKENS_TABLE)

if BQ_FULL_INTERVIEWS_TABLE is not None:
    BQ_FULL_TABLE_RESOLVED = normalize_bq_table_name(BQ_FULL_INTERVIEWS_TABLE)
else:
    other_tables = catalog[catalog["table_fq"] != BQ_TOKENS_TABLE_RESOLVED].copy()
    if len(other_tables) == 0:
        raise RuntimeError(
            "Só encontrei a tabela de chunks/tokens no dataset. "
            "Preencha BQ_FULL_INTERVIEWS_TABLE manualmente com a tabela de entrevistas completas."
        )
    if len(other_tables) == 1:
        BQ_FULL_TABLE_RESOLVED = other_tables.iloc[0]["table_fq"]
        print("Como só existe uma outra tabela no dataset, ela foi escolhida como entrevistas completas:")
        print("  ", BQ_FULL_TABLE_RESOLVED)
    else:
        def full_table_score(schema_names, table_id):
            names = " ".join(str(x).lower() for x in (schema_names or []))
            table_id_l = str(table_id).lower()
            score = 0
            score += 5 * any(term in table_id_l for term in ["completa", "completo", "full", "raw", "transcricao", "transcricoes", "entrevistas"])
            score -= 6 * any(term in table_id_l for term in ["chunk", "token", "segment"])
            score += 4 * any(term in names for term in ["transcricao", "transcrição", "texto", "text", "full_text", "conteudo", "content"])
            score -= 4 * any(term in names for term in ["chunk_id", "chunk_index", "tokens"])
            return score

        other_tables["full_table_score"] = other_tables.apply(lambda r: full_table_score(r.get("schema_names"), r.get("table_id")), axis=1)
        ranked = other_tables.sort_values(["full_table_score", "num_rows"], ascending=[False, False])
        save_df(ranked[["table_id", "table_fq", "num_rows", "num_bytes", "full_table_score", "schema_names"]], "00_bq_candidatas_entrevistas_completas.csv")
        display(ranked[["table_id", "table_fq", "num_rows", "full_table_score", "schema_names"]])
        BQ_FULL_TABLE_RESOLVED = ranked.iloc[0]["table_fq"]
        print("Tabela escolhida automaticamente como entrevistas completas:")
        print("  ", BQ_FULL_TABLE_RESOLVED)
        print("Se estiver errado, preencha BQ_FULL_INTERVIEWS_TABLE na seção 2 e rode novamente.")

print("\nTabelas que serão baixadas:")
print("TOKENS/CHUNKS       :", BQ_TOKENS_TABLE_RESOLVED)
print("ENTREVISTAS COMPLETAS:", BQ_FULL_TABLE_RESOLVED)

df_tokens = read_bq_table(client, BQ_TOKENS_TABLE_RESOLVED, limit_rows=BQ_LIMIT_ROWS, use_cache=USE_BQ_CACHE)
df_full = read_bq_table(client, BQ_FULL_TABLE_RESOLVED, limit_rows=BQ_LIMIT_ROWS, use_cache=USE_BQ_CACHE)

print("\nShapes carregados:")
print("df_tokens:", df_tokens.shape)
print("df_full  :", df_full.shape)


Salvo: data/diagnostics/00_bq_catalogo_tabelas.csv | shape=(2, 7)


,table_id,table_fq,num_rows,num_bytes,modified,schema_names
0,entrevistas_chunks_v2,kyra-pml-20261.kyra_base_2026_1.entrevistas_chunks_v2,12041,17419281,2026-03-23 00:22:28.058000+00:00,"[doc_id, chunk_id, chunk_index, source_filename_hash, collection_batch, mes, ano, projeto, pais, marca_foco, publico, tipo_sessao, canal_origem, contexto_origem, idioma, texto_analise, n_caractere..."
1,entrevistas_docs_v2,kyra-pml-20261.kyra_base_2026_1.entrevistas_docs_v2,155,12882793,2026-03-23 00:19:40.709000+00:00,"[doc_id, source_filename_hash, collection_batch, mes, mes_num, ano, projeto, pais, marca_foco, publico, tipo_sessao, canal_origem, contexto_origem, idioma, data_bruta_nome_arquivo, hora_bruta_nome..."


Como só existe uma outra tabela no dataset, ela foi escolhida como entrevistas completas:
   kyra-pml-20261.kyra_base_2026_1.entrevistas_docs_v2

Tabelas que serão baixadas:
TOKENS/CHUNKS       : kyra-pml-20261.kyra_base_2026_1.entrevistas_chunks_v2
ENTREVISTAS COMPLETAS: kyra-pml-20261.kyra_base_2026_1.entrevistas_docs_v2
Rodando query: SELECT * FROM `kyra-pml-20261.kyra_base_2026_1.entrevistas_chunks_v2`
Snapshot salvo em: data/raw/bq_snapshots/kyra_pml_20261__kyra_base_2026_1__entrevistas_chunks_v2.parquet | shape=(12041, 32)
Rodando query: SELECT * FROM `kyra-pml-20261.kyra_base_2026_1.entrevistas_docs_v2`
Snapshot salvo em: data/raw/bq_snapshots/kyra_pml_20261__kyra_base_2026_1__entrevistas_docs_v2.parquet | shape=(155, 29)

Shapes carregados:
df_tokens: (12041, 32)
df_full  : (155, 29)


## 5. Perfil estrutural das bases


In [5]:
profile_tokens = dataframe_profile(df_tokens, "tokens_chunks")
profile_full = dataframe_profile(df_full, "entrevistas_completas")
profile = pd.concat([profile_tokens, profile_full], ignore_index=True)
save_df(profile, "01_perfil_colunas.csv")

display(profile_tokens)
display(profile_full)

cols_tokens = guess_columns(df_tokens)
cols_full = guess_columns(df_full)

cols_detectadas = pd.DataFrame([
    {"base": "tokens_chunks", **cols_tokens},
    {"base": "entrevistas_completas", **cols_full},
])
save_df(cols_detectadas, "01_colunas_detectadas.csv")
save_json({"tokens_chunks": cols_tokens, "entrevistas_completas": cols_full}, "01_colunas_detectadas.json")
display(cols_detectadas)

if cols_tokens.get("text") is None:
    raise RuntimeError("Não consegui detectar coluna de texto na base de chunks/tokens. Veja 01_perfil_colunas.csv e ajuste manualmente cols_tokens['text'].")
if cols_full.get("text") is None:
    raise RuntimeError("Não consegui detectar coluna de texto na base de entrevistas completas. Veja 01_perfil_colunas.csv e ajuste manualmente cols_full['text'].")

text_col_tok = cols_tokens["text"]
text_col_full = cols_full["text"]
print("Coluna texto chunks/tokens       :", text_col_tok)
print("Coluna texto entrevistas completas:", text_col_full)


Salvo: data/diagnostics/01_perfil_colunas.csv | shape=(61, 9)


,base,coluna,dtype,n_linhas,n_na,pct_na,n_unique,pct_unique_sobre_nao_nulo,amostras
0,tokens_chunks,doc_id,object,12041,0,0.0,155,0.0129,doc_b53730e47cc798ef | doc_b53730e47cc798ef | doc_b53730e47cc798ef
1,tokens_chunks,chunk_id,object,12041,0,0.0,12041,1.0000,doc_b53730e47cc798ef_ch_0001 | doc_b53730e47cc798ef_ch_0002 | doc_b53730e47cc798ef_ch_0003
2,tokens_chunks,chunk_index,Int64,12041,0,0.0,156,0.0130,1 | 2 | 3
3,tokens_chunks,source_filename_hash,object,12041,0,0.0,155,0.0129,e108bde61f76578b40a5 | e108bde61f76578b40a5 | e108bde61f76578b40a5
4,tokens_chunks,collection_batch,object,12041,0,0.0,1,0.0001,v2 | v2 | v2
5,tokens_chunks,mes,object,12041,0,0.0,8,0.0007,abril | abril | abril
6,tokens_chunks,ano,object,12041,12041,1.0,0,NaN,
7,tokens_chunks,projeto,object,12041,0,0.0,12,0.0010,biome | biome | biome
8,tokens_chunks,pais,object,12041,0,0.0,4,0.0003,nao_inferido | nao_inferido | nao_inferido
9,tokens_chunks,marca_foco,object,12041,0,0.0,3,0.0002,nao_inferido | nao_inferido | nao_inferido


,base,coluna,dtype,n_linhas,n_na,pct_na,n_unique,pct_unique_sobre_nao_nulo,amostras
0,entrevistas_completas,doc_id,object,155,0,0.0000,155,1.0000,doc_b53730e47cc798ef | doc_9ed820073ce06c76 | doc_a11e5233b2bda4e2
1,entrevistas_completas,source_filename_hash,object,155,0,0.0000,155,1.0000,e108bde61f76578b40a5 | 612b6b60de6e532a1a4b | d5651dd4c490d22f337f
2,entrevistas_completas,collection_batch,object,155,0,0.0000,1,0.0065,v2 | v2 | v2
3,entrevistas_completas,mes,object,155,0,0.0000,8,0.0516,abril | abril | abril
4,entrevistas_completas,mes_num,Int64,155,0,0.0000,8,0.0516,4 | 4 | 4
5,entrevistas_completas,ano,object,155,155,1.0000,0,NaN,
6,entrevistas_completas,projeto,object,155,0,0.0000,12,0.0774,biome | biome | biome
7,entrevistas_completas,pais,object,155,0,0.0000,4,0.0258,nao_inferido | nao_inferido | nao_inferido
8,entrevistas_completas,marca_foco,object,155,0,0.0000,3,0.0194,nao_inferido | nao_inferido | nao_inferido
9,entrevistas_completas,publico,object,155,0,0.0000,4,0.0258,nao_inferido | nao_inferido | nao_inferido


Salvo: data/diagnostics/01_colunas_detectadas.csv | shape=(2, 8)
Salvo: data/diagnostics/01_colunas_detectadas.json


,base,doc_id,chunk_id,chunk_index,language,text,tokens,date
0,tokens_chunks,doc_id,chunk_id,chunk_index,idioma,texto_analise,n_palavras,timestamp_inicial
1,entrevistas_completas,doc_id,None,None,idioma,texto_analise,n_palavras_total,data_bruta_nome_arquivo


Coluna texto chunks/tokens       : texto_analise
Coluna texto entrevistas completas: texto_analise


## 6. Amostras e estatísticas básicas de texto


In [6]:
def text_stats_series(s: pd.Series) -> pd.DataFrame:
    txt = s.fillna("").astype(str)
    out = pd.DataFrame({
        "n_chars": txt.str.len(),
        "n_words": txt.apply(lambda x: len(simple_word_tokens(x))),
        "n_digits": txt.str.count(r"\d"),
        "n_punct": txt.str.count(r"[^\w\s]"),
        "n_newlines": txt.str.count(r"\n"),
    })
    out["digit_ratio_chars"] = np.where(out["n_chars"] > 0, out["n_digits"] / out["n_chars"], 0)
    out["punct_ratio_chars"] = np.where(out["n_chars"] > 0, out["n_punct"] / out["n_chars"], 0)
    return out

stats_tok = text_stats_series(df_tokens[text_col_tok])
stats_full = text_stats_series(df_full[text_col_full])

summary_rows = []
for name, st in [("tokens_chunks", stats_tok), ("entrevistas_completas", stats_full)]:
    desc = st.describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).T.reset_index().rename(columns={"index": "metrica"})
    desc.insert(0, "base", name)
    summary_rows.append(desc)
summary_text = pd.concat(summary_rows, ignore_index=True)
save_df(summary_text, "02_estatisticas_texto_resumo.csv")
display(summary_text)

# Salva amostras para leitura manual
sample_cols_tok = [c for c in [cols_tokens.get("doc_id"), cols_tokens.get("chunk_id"), cols_tokens.get("chunk_index"), cols_tokens.get("language"), text_col_tok] if c and c in df_tokens.columns]
sample_cols_full = [c for c in [cols_full.get("doc_id"), cols_full.get("language"), text_col_full] if c and c in df_full.columns]

sample_tokens = df_tokens[sample_cols_tok].sample(min(100, len(df_tokens)), random_state=42) if len(df_tokens) else pd.DataFrame()
sample_full = df_full[sample_cols_full].sample(min(50, len(df_full)), random_state=42) if len(df_full) else pd.DataFrame()

save_df(sample_tokens, "02_amostra_chunks_tokens.csv")
save_df(sample_full, "02_amostra_entrevistas_completas.csv")

display(sample_tokens.head(10))
display(sample_full.head(5))


Salvo: data/diagnostics/02_estatisticas_texto_resumo.csv | shape=(14, 16)


,base,metrica,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
0,tokens_chunks,n_chars,12041.0,1187.861473,68.841017,153.000000,1015.800000,1185.000000,1188.000000,1193.000000,1196.000000,1198.000000,1199.000000,1199.000000,1199.000000,1199.000000
1,tokens_chunks,n_words,12041.0,161.498131,22.466772,17.000000,103.400000,124.000000,133.000000,148.000000,165.000000,177.000000,187.000000,192.000000,200.000000,222.000000
2,tokens_chunks,n_digits,12041.0,62.708081,34.875875,0.000000,5.000000,13.000000,20.000000,37.000000,58.000000,84.000000,113.000000,129.000000,152.600000,203.000000
3,tokens_chunks,n_punct,12041.0,60.447471,14.038175,7.000000,28.000000,38.000000,43.000000,51.000000,60.000000,70.000000,79.000000,85.000000,94.000000,120.000000
4,tokens_chunks,n_newlines,12041.0,57.552612,24.859858,5.000000,17.000000,23.000000,26.000000,38.000000,55.000000,75.000000,93.000000,103.000000,118.000000,159.000000
5,tokens_chunks,digit_ratio_chars,12041.0,0.053185,0.030086,0.000000,0.004195,0.010870,0.016708,0.030962,0.048495,0.071846,0.096074,0.109349,0.131150,0.222222
6,tokens_chunks,punct_ratio_chars,12041.0,0.051022,0.011819,0.005853,0.025105,0.032581,0.036759,0.042749,0.050251,0.058577,0.066834,0.071609,0.079750,0.100251
7,entrevistas_completas,n_chars,155.0,80953.335484,38314.177836,1206.000000,4530.240000,24477.100000,31210.200000,52177.500000,78768.000000,114498.500000,132793.800000,135436.700000,145022.940000,162745.000000
8,entrevistas_completas,n_words,155.0,10980.161290,4882.976273,146.000000,555.400000,3457.900000,4234.600000,7392.000000,10966.000000,15449.500000,17424.200000,18022.600000,18965.340000,19035.000000
9,entrevistas_completas,n_digits,155.0,4277.600000,3182.531439,76.000000,315.800000,816.100000,1091.000000,1665.500000,3051.000000,6470.500000,9582.800000,10459.700000,11433.520000,13865.000000


Salvo: data/diagnostics/02_amostra_chunks_tokens.csv | shape=(100, 5)
Salvo: data/diagnostics/02_amostra_entrevistas_completas.csv | shape=(50, 3)


,doc_id,chunk_id,chunk_index,idioma,texto_analise
8872,doc_981214beda5a6637,doc_981214beda5a6637_ch_0015,15,pt,"as lojas e ele já tinha ido fazer pré-trabalho antes, então ele já sabia onde ele queria ir E aí ele \nfalou que ele já tinha ido lá durante a semana, e aí a gente entrou com ele na Besni, ele ent..."
3780,doc_ca13ed7a8ce3eb34,doc_ca13ed7a8ce3eb34_ch_0032,32,pt,"gundo nomezinho dele, informando qual foi a edição dele. É por isso \nque a gente sempre fala tradicional, porque é o primeiro. \n \n \n \n \n \nSpeaker 2 - 52:36\n \n Tradicional significa primei..."
7528,doc_52108ac94ef635f9,doc_52108ac94ef635f9_ch_0036,36,pt,"- 43:41\n \nMeeting Title: EP12_Consultora_3Cs.m4a Meeting created at: 20th Mar, 2026 - 3:09 PM\n29 / 44\n \n Entendeu? Igual essa amostra do... Os três o quê? Desculpa, Pri, conta para a gente di..."
2619,doc_d2d8ffad5d07749d,doc_d2d8ffad5d07749d_ch_0104,104,es,"Speaker 7 - 01:46:29\n \n Bueno, me gusta porque no se pone en un tono muy oscuro, un tono clarito, traen como que la gama que puede \nhaber dentro de esos dos tonos y además que el maquillaje se ..."
8517,doc_44982372dd74c6da,doc_44982372dd74c6da_ch_0011,11,pt,"orque sou só eu de homem \naqui. É muita mulher. Você não tem noção. É mãe, namorada, tia, é duas prima. A prima tem duas filhas que é \nprima. As dogs também é tudo fêmea. Então praticamente sou ..."
3325,doc_eeb118edb6d745e0,doc_eeb118edb6d745e0_ch_0050,50,pt,"e ela falou da experimentação? Ela falou, \ngente, e ela mesmo usa muito. E ela fala, ela vai viajar, vai ver os parentes. Tipo, uma mala inteira. Ela leva só de \nprodutos. \n \n \n \n \n \nSpeak..."
5206,doc_ecbea0911b71e6fa,doc_ecbea0911b71e6fa_ch_0027,27,pt,"Speaker 3 - 27:01\n \n Hidratado, mancha, as manchas no meu corpo clareou. Sabe, ele não me deixa ruxo. O meu pé, eu tô passando no \nmeu pé, se eu escrevo, e melhorou no meu pé, sabe? Eu gosto mu..."
11571,doc_3a9c3f93806290fb,doc_3a9c3f93806290fb_ch_0001,1,pt,"Speaker 1 - 00:04\n \n Olá, tudo bem? Que bom ver você por aqui. Nós somos Akira, uma empresa dedicada a entender \ncomportamentos, necessidades e desejos das pessoas. Assim, ajudamos diferentes m..."
3706,doc_fcc334ae6da3d7f2,doc_fcc334ae6da3d7f2_ch_0007,7,pt,"ha visto isso, nunca tinha reparado, mas o que me chamou \nmuito a atenção foi essa coisa do presente muito bem trabalhado, com muito destaque na loja deles. E na nossa, \nenfim, fazendo aqui toda..."
10295,doc_4b466654cd374bdf,doc_4b466654cd374bdf_ch_0073,73,pt,"Speaker 2 - 57:37\n \n Não, mas você compara que uma empresa quando resolve ter uma fusão com a outra, isso já tá sendo estudado, \nné? Então prepara as pessoas antes, depois tu. \n \n \n \n \nSpe..."


,doc_id,idioma,texto_analise
81,doc_a966111a7b9eb0a6,pt,"Speaker 1 - 00:00\n \n Então, tá bom. Bom. \n \n \n \n \n \nSpeaker 2 - 00:47\n \n Dia! \n \n \n \n \n \nSpeaker 3 - 00:48\n \n Bom dia, menina! Tudo bem com vocês? \n \n \n \n \n \nSpeaker 2 - 00..."
142,doc_6e02133bb486de22,pt,"Speaker 1 - 00:04\n \n Olá, tudo bem? Que bom ver você por aqui. Nós somos Akira, uma empresa dedicada a entender \ncomportamentos, necessidades e desejos das pessoas. Assim, ajudamos diferentes m..."
31,doc_ab0b946958f4f005,pt,"Speaker 1 - 05:07\n \n Oi meninas, você tá me ouvindo? \n \n \n \n \n \nSpeaker 2 - 05:11\n \n Sim, oi Louise. Oi, tudo bom? \n \n \n \n \n \nSpeaker 3 - 05:18\n \n Oi Célia, tudo bem? \n \n \n \n..."
29,doc_33640fb8f77ec4a8,pt,"Speaker 1 - 00:17\n \n Por enquanto, ninguém. Mari, vou chamar no grupo, tá? \n \n \n \n \n \nSpeaker 2 - 00:21\n \n Tá bom, vou esperar aqui. \n \n \n \n \nSpeaker 1 - 02:21\n \n Oi, André! Bom d..."
118,doc_500a84cf9936a539,pt,"Speaker 1 - 03:31\n \n Muito obrigada. \n \n \n \n \n \nSpeaker 2 - 04:01\n \n Certo, e que tu chegasse a tempo, que tu fizesse uma boa viagem, aí não está chovendo, aqui está bem frio, tem \nhora..."


## 7. Cobertura de IDs e reconciliação entre chunks e entrevistas completas


In [7]:
doc_tok = cols_tokens.get("doc_id")
doc_full = cols_full.get("doc_id")

coverage_rows = []
if doc_tok and doc_tok in df_tokens.columns:
    coverage_rows.append({
        "base": "tokens_chunks",
        "coluna_id": doc_tok,
        "linhas": len(df_tokens),
        "ids_unicos": df_tokens[doc_tok].nunique(dropna=True),
        "ids_nulos": int(df_tokens[doc_tok].isna().sum()),
    })
if doc_full and doc_full in df_full.columns:
    coverage_rows.append({
        "base": "entrevistas_completas",
        "coluna_id": doc_full,
        "linhas": len(df_full),
        "ids_unicos": df_full[doc_full].nunique(dropna=True),
        "ids_nulos": int(df_full[doc_full].isna().sum()),
    })

coverage = pd.DataFrame(coverage_rows)
save_df(coverage, "03_cobertura_ids.csv")
display(coverage)

if doc_tok and doc_full and doc_tok in df_tokens.columns and doc_full in df_full.columns:
    ids_tok = set(df_tokens[doc_tok].dropna().astype(str))
    ids_full = set(df_full[doc_full].dropna().astype(str))
    id_cmp = pd.DataFrame([{
        "ids_chunks": len(ids_tok),
        "ids_full": len(ids_full),
        "ids_intersecao": len(ids_tok & ids_full),
        "ids_so_chunks": len(ids_tok - ids_full),
        "ids_so_full": len(ids_full - ids_tok),
        "pct_chunks_cobertos_full": round(len(ids_tok & ids_full) / len(ids_tok), 4) if ids_tok else None,
        "pct_full_cobertos_chunks": round(len(ids_tok & ids_full) / len(ids_full), 4) if ids_full else None,
    }])
    save_df(id_cmp, "03_reconciliacao_ids_resumo.csv")
    display(id_cmp)

    pd.DataFrame({"doc_id_so_chunks": sorted(list(ids_tok - ids_full))[:5000]}).to_csv(DIAG_DIR / "03_ids_so_chunks_amostra.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame({"doc_id_so_full": sorted(list(ids_full - ids_tok))[:5000]}).to_csv(DIAG_DIR / "03_ids_so_full_amostra.csv", index=False, encoding="utf-8-sig")
else:
    print("Não foi possível reconciliar IDs automaticamente. Veja 01_colunas_detectadas.csv.")


Salvo: data/diagnostics/03_cobertura_ids.csv | shape=(2, 5)


,base,coluna_id,linhas,ids_unicos,ids_nulos
0,tokens_chunks,doc_id,12041,155,0
1,entrevistas_completas,doc_id,155,155,0


Salvo: data/diagnostics/03_reconciliacao_ids_resumo.csv | shape=(1, 7)


,ids_chunks,ids_full,ids_intersecao,ids_so_chunks,ids_so_full,pct_chunks_cobertos_full,pct_full_cobertos_chunks
0,155,155,155,0,0,1.0,1.0


## 8. Diagnóstico de segmentação, ordenação e overlap

Esta célula substitui a célula que dava erro no notebook anterior. Ela **não usa `getattr`** em linhas do pandas e **não depende de coluna interna inexistente**.


In [8]:
def token_set_for_overlap(text: str) -> set:
    toks = re.findall(r"[a-z]{3,}", text_for_hash(text))
    return set(toks)

seg_rows = []
doc_col = cols_tokens.get("doc_id")
idx_col = cols_tokens.get("chunk_index")
chunk_col = cols_tokens.get("chunk_id")

if doc_col and doc_col in df_tokens.columns:
    tmp_seg = df_tokens.copy()
    tmp_seg["row_pos_diag"] = np.arange(len(tmp_seg))
    if idx_col and idx_col in tmp_seg.columns:
        tmp_seg["chunk_index_numeric_diag"] = pd.to_numeric(tmp_seg[idx_col], errors="coerce")
        sort_cols = [doc_col, "chunk_index_numeric_diag", "row_pos_diag"]
    else:
        tmp_seg["chunk_index_numeric_diag"] = np.nan
        sort_cols = [doc_col, "row_pos_diag"]

    for doc_id_value, group_df in tmp_seg.sort_values(sort_cols).groupby(doc_col, dropna=False):
        numeric_indices = group_df["chunk_index_numeric_diag"].dropna().astype(int).tolist()
        expected = set(range(min(numeric_indices), max(numeric_indices) + 1)) if numeric_indices else set()
        observed = set(numeric_indices)
        text_lengths = group_df[text_col_tok].fillna("").astype(str).apply(lambda x: len(simple_word_tokens(x)))
        seg_rows.append({
            "doc_id": doc_id_value,
            "n_chunks": len(group_df),
            "min_chunk_index": min(numeric_indices) if numeric_indices else None,
            "max_chunk_index": max(numeric_indices) if numeric_indices else None,
            "n_missing_indices": len(expected - observed),
            "missing_indices_sample": sorted(list(expected - observed))[:30],
            "n_duplicate_indices": int(pd.Series(numeric_indices).duplicated().sum()) if numeric_indices else 0,
            "mean_words_per_chunk": round(float(text_lengths.mean()), 2) if len(text_lengths) else None,
            "p10_words_per_chunk": round(float(text_lengths.quantile(.10)), 2) if len(text_lengths) else None,
            "p90_words_per_chunk": round(float(text_lengths.quantile(.90)), 2) if len(text_lengths) else None,
        })

    seg_df = pd.DataFrame(seg_rows)
    save_df(seg_df, "04_segmentacao_por_documento.csv")
    display(seg_df.describe(include="all").T)
else:
    seg_df = pd.DataFrame()
    print("doc_id não detectado na base de chunks; diagnóstico de segmentação por documento limitado.")

# Duplicatas exatas ou quase exatas de texto normalizado
normalized_text_for_dups = df_tokens[text_col_tok].fillna("").astype(str).apply(text_for_hash)
dup_text_mask = normalized_text_for_dups.duplicated(keep=False) & normalized_text_for_dups.ne("")
dup_text = df_tokens.loc[dup_text_mask].copy()
if len(dup_text):
    dup_text["texto_normalizado_para_duplicidade"] = normalized_text_for_dups.loc[dup_text_mask].values
    keep_cols = [c for c in [doc_col, chunk_col, idx_col, text_col_tok, "texto_normalizado_para_duplicidade"] if c and c in dup_text.columns]
    save_df(dup_text[keep_cols].head(1000), "04_chunks_texto_duplicado_amostra.csv")
    print(f"Chunks com texto normalizado duplicado: {len(dup_text):,}")
else:
    print("Nenhum texto de chunk exatamente duplicado após normalização básica.")

# Overlap entre chunks consecutivos
pairs = []
MAX_PAIRS_FOR_OVERLAP = 200_000
n_pairs = 0

if doc_col and doc_col in df_tokens.columns:
    base_cols = [doc_col, text_col_tok]
    if idx_col and idx_col in df_tokens.columns:
        base_cols.append(idx_col)
    if chunk_col and chunk_col in df_tokens.columns:
        base_cols.append(chunk_col)

    tmp_overlap = df_tokens[base_cols].copy()
    tmp_overlap["row_pos_diag"] = np.arange(len(tmp_overlap))
    if idx_col and idx_col in tmp_overlap.columns:
        tmp_overlap["chunk_index_numeric_diag"] = pd.to_numeric(tmp_overlap[idx_col], errors="coerce")
        tmp_overlap = tmp_overlap.sort_values([doc_col, "chunk_index_numeric_diag", "row_pos_diag"])
    else:
        tmp_overlap = tmp_overlap.sort_values([doc_col, "row_pos_diag"])

    for doc_id_value, sub_df in tmp_overlap.groupby(doc_col, dropna=False):
        records = sub_df.to_dict("records")
        for rec_a, rec_b in zip(records[:-1], records[1:]):
            set_a = token_set_for_overlap(rec_a.get(text_col_tok, ""))
            set_b = token_set_for_overlap(rec_b.get(text_col_tok, ""))
            union = len(set_a | set_b)
            jac = len(set_a & set_b) / union if union else 0
            pairs.append({
                "doc_id": doc_id_value,
                "chunk_index_a": rec_a.get(idx_col) if idx_col else None,
                "chunk_index_b": rec_b.get(idx_col) if idx_col else None,
                "chunk_id_a": rec_a.get(chunk_col) if chunk_col else None,
                "chunk_id_b": rec_b.get(chunk_col) if chunk_col else None,
                "jaccard_tokens_consecutivos": round(jac, 4),
                "n_tokens_a": len(set_a),
                "n_tokens_b": len(set_b),
            })
            n_pairs += 1
            if n_pairs >= MAX_PAIRS_FOR_OVERLAP:
                break
        if n_pairs >= MAX_PAIRS_FOR_OVERLAP:
            break

    overlap_df = pd.DataFrame(pairs)
    save_df(overlap_df, "04_overlap_chunks_consecutivos.csv")
    if len(overlap_df):
        display(overlap_df["jaccard_tokens_consecutivos"].describe(percentiles=[.5, .75, .9, .95, .99]).to_frame().T)
        high_overlap = overlap_df[overlap_df["jaccard_tokens_consecutivos"] >= 0.75].sort_values("jaccard_tokens_consecutivos", ascending=False)
        save_df(high_overlap.head(1000), "04_overlap_alto_chunks_consecutivos.csv")
else:
    overlap_df = pd.DataFrame()
    print("Sem doc_id: overlap consecutivo por entrevista não calculado.")


Salvo: data/diagnostics/04_segmentacao_por_documento.csv | shape=(155, 10)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
doc_id,155,155,doc_012fe174842b390e,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_chunks,155.0,NaN,NaN,NaN,77.683871,36.63292,2.0,50.5,76.0,109.5,156.0
min_chunk_index,155.0,NaN,NaN,NaN,1.0,0.0,1.0,1.0,1.0,1.0,1.0
max_chunk_index,155.0,NaN,NaN,NaN,77.683871,36.63292,2.0,50.5,76.0,109.5,156.0
n_missing_indices,155.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
missing_indices_sample,155,1,[],155,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_duplicate_indices,155.0,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean_words_per_chunk,155.0,NaN,NaN,NaN,162.861806,13.440944,82.0,157.185,164.71,171.795,191.34
p10_words_per_chunk,155.0,NaN,NaN,NaN,142.816129,16.591683,31.6,133.5,146.0,153.65,180.3
p90_words_per_chunk,155.0,NaN,NaN,NaN,182.187097,11.296102,124.6,177.0,184.0,189.8,202.0


Salvo: data/diagnostics/04_chunks_texto_duplicado_amostra.csv | shape=(27, 5)
Chunks com texto normalizado duplicado: 27
Salvo: data/diagnostics/04_overlap_chunks_consecutivos.csv | shape=(11886, 8)


,count,mean,std,min,50%,75%,90%,95%,99%,max
jaccard_tokens_consecutivos,11886.0,0.250215,0.048514,0.0707,0.2479,0.2815,0.3137,0.3333,0.375,0.4898


Salvo: data/diagnostics/04_overlap_alto_chunks_consecutivos.csv | shape=(0, 8)


## 9. Diagnóstico de idioma e mistura PT/ES/EN

A ideia aqui não é fazer classificação perfeita de idioma, mas encontrar sinais de erro de segmentação e limpeza: chunks em idioma errado, mistura de idiomas e excesso de palavras funcionais.


In [9]:
PT_FUNCTION_WORDS = set("""
a o as os um uma uns umas de da das do dos em no na nos nas para por com sem sob sobre entre ate após antes depois
que quem qual quais quando quanto onde como porque pois se e ou mas porem entretanto contudo entao também nao nem sim ser estar ter haver foi era sao será tinha tenho voce voces eu ele ela eles elas meu minha seu sua nosso nossa isso isto aquilo
""".split())

ES_FUNCTION_WORDS = set("""
el la los las un una unos unas de del en por para con sin sobre entre hasta despues antes que quien cual cuales cuando cuanto donde como porque pues si y o pero aunque entonces tambien no ni ser estar tener haber fue era son sera tenia tengo usted ustedes yo el ella ellos ellas mi mis su sus nuestro nuestra esto eso aquello
""".split())

EN_FUNCTION_WORDS = set("""
the a an and or but if then else when where what which who whom whose why how is are was were be been being have has had do does did not no yes in on at by for from with without about between after before this that these those i you he she it we they my your his her our their
""".split())

ALL_FUNCTION_WORDS = PT_FUNCTION_WORDS | ES_FUNCTION_WORDS | EN_FUNCTION_WORDS


def detect_language_light(text: str) -> str:
    toks = simple_word_tokens(text)
    if not toks:
        return "unknown"
    counts = Counter(toks)
    scores = {
        "pt": sum(counts[t] for t in PT_FUNCTION_WORDS),
        "es": sum(counts[t] for t in ES_FUNCTION_WORDS),
        "en": sum(counts[t] for t in EN_FUNCTION_WORDS),
    }
    best_lang, best_score = max(scores.items(), key=lambda kv: kv[1])
    total_function = sum(scores.values())
    if best_score < 2:
        return "unknown"
    # Se segunda língua está muito próxima, marca misto.
    ordered = sorted(scores.values(), reverse=True)
    if len(ordered) >= 2 and ordered[1] >= 0.45 * ordered[0] and total_function >= 8:
        return "mixed"
    return best_lang

lang_col = cols_tokens.get("language")
lang_audit = df_tokens[[text_col_tok]].copy()
if lang_col and lang_col in df_tokens.columns:
    lang_audit["idioma_original"] = df_tokens[lang_col].astype(str)
else:
    lang_audit["idioma_original"] = None
lang_audit["idioma_detectado_light"] = df_tokens[text_col_tok].fillna("").astype(str).apply(detect_language_light)

# Scores detalhados
score_rows = []
for text in df_tokens[text_col_tok].fillna("").astype(str):
    toks = simple_word_tokens(text)
    counts = Counter(toks)
    n_words = len(toks)
    pt_n = sum(counts[t] for t in PT_FUNCTION_WORDS)
    es_n = sum(counts[t] for t in ES_FUNCTION_WORDS)
    en_n = sum(counts[t] for t in EN_FUNCTION_WORDS)
    score_rows.append({
        "n_words": n_words,
        "pt_function_words": pt_n,
        "es_function_words": es_n,
        "en_function_words": en_n,
        "function_word_ratio": round((pt_n + es_n + en_n) / n_words, 4) if n_words else 0,
    })
lang_scores = pd.DataFrame(score_rows)
lang_audit = pd.concat([lang_audit, lang_scores], axis=1)

save_df(lang_audit, "05_idioma_auditoria_por_chunk.csv")

dist_lang = lang_audit["idioma_detectado_light"].value_counts(dropna=False).reset_index()
dist_lang.columns = ["idioma_detectado_light", "n_chunks"]
save_df(dist_lang, "05_idioma_distribuicao_detectada.csv")
display(dist_lang)

if lang_col and lang_col in df_tokens.columns:
    mismatch = lang_audit[
        lang_audit["idioma_detectado_light"].isin(["pt", "es", "en", "mixed"])
        & lang_audit["idioma_original"].notna()
        & (lang_audit["idioma_original"].str.lower().str[:2] != lang_audit["idioma_detectado_light"].str[:2])
    ].copy()
    mismatch_cols = [c for c in [doc_col, chunk_col, idx_col, lang_col, text_col_tok] if c and c in df_tokens.columns]
    mismatch_export = df_tokens.loc[mismatch.index, mismatch_cols].copy() if len(mismatch) else pd.DataFrame(columns=mismatch_cols)
    if len(mismatch_export):
        mismatch_export["idioma_detectado_light"] = mismatch["idioma_detectado_light"].values
    save_df(mismatch_export.head(1000), "05_idioma_mismatch_amostra.csv")
    print("Mismatches idioma original vs detectado:", len(mismatch_export))

mixed_idx = lang_audit[lang_audit["idioma_detectado_light"].eq("mixed")].index
mixed_cols = [c for c in [doc_col, chunk_col, idx_col, lang_col, text_col_tok] if c and c in df_tokens.columns]
save_df(df_tokens.loc[mixed_idx, mixed_cols].head(1000), "05_idioma_misto_amostra.csv")


Salvo: data/diagnostics/05_idioma_auditoria_por_chunk.csv | shape=(12041, 8)
Salvo: data/diagnostics/05_idioma_distribuicao_detectada.csv | shape=(4, 2)


,idioma_detectado_light,n_chunks
0,pt,8628
1,mixed,3138
2,es,238
3,en,37


Salvo: data/diagnostics/05_idioma_mismatch_amostra.csv | shape=(1000, 6)
Mismatches idioma original vs detectado: 3175
Salvo: data/diagnostics/05_idioma_misto_amostra.csv | shape=(1000, 5)


PosixPath('/Users/emanuelgandra/Desktop/Faculdade/6Período/ProjetoCienciaDados/Kyra/kyrav2/data/diagnostics/05_idioma_misto_amostra.csv')

## 10. Diagnóstico de tokens, preposições, fillers e vocabulário dominante


In [10]:
token_col = cols_tokens.get("tokens")

# Se existir coluna de tokens/lista, usa ela; se não, tokeniza a coluna de texto para diagnóstico.
all_token_lists = []
source_token_mode = None

if token_col and token_col in df_tokens.columns:
    parsed = df_tokens[token_col].apply(parse_possible_list)
    parse_rate = parsed.notna().mean()
    if parse_rate >= 0.20:
        all_token_lists = parsed.fillna('').apply(lambda x: x if isinstance(x, list) else []).tolist()
        source_token_mode = f"coluna_existente:{token_col}"

if source_token_mode is None:
    all_token_lists = df_tokens[text_col_tok].fillna("").astype(str).apply(simple_word_tokens).tolist()
    source_token_mode = f"tokenizacao_diagnostica_a_partir_de:{text_col_tok}"

print("Fonte dos tokens para diagnóstico:", source_token_mode)

flat_tokens = []
for lst in all_token_lists:
    for tok in lst:
        if tok is None:
            continue
        tok_norm = strip_accents_lower(str(tok))
        tok_norm = re.sub(r"[^a-z0-9]+", "", tok_norm)
        if tok_norm:
            flat_tokens.append(tok_norm)

tok_counts = Counter(flat_tokens)
top_tokens = pd.DataFrame(tok_counts.most_common(5000), columns=["token", "freq"])
top_tokens["pct_sobre_total_tokens"] = top_tokens["freq"] / max(1, len(flat_tokens))
top_tokens["is_function_word"] = top_tokens["token"].isin(ALL_FUNCTION_WORDS)
top_tokens["is_short_le_2"] = top_tokens["token"].str.len() <= 2
save_df(top_tokens, "06_tokens_top5000.csv")
display(top_tokens.head(100))

function_top = top_tokens[top_tokens["is_function_word"]].head(1000)
save_df(function_top, "06_tokens_funcionais_top1000.csv")
display(function_top.head(100))

# Métricas por chunk de densidade de tokens funcionais
fw_ratios = []
for lst in all_token_lists:
    clean = []
    for tok in lst:
        tok_norm = strip_accents_lower(str(tok))
        tok_norm = re.sub(r"[^a-z0-9]+", "", tok_norm)
        if tok_norm:
            clean.append(tok_norm)
    n = len(clean)
    fw = sum(t in ALL_FUNCTION_WORDS for t in clean)
    short = sum(len(t) <= 2 for t in clean)
    unique = len(set(clean))
    fw_ratios.append({
        "n_tokens_diag": n,
        "n_tokens_unicos_diag": unique,
        "token_unique_ratio_diag": round(unique / n, 4) if n else 0,
        "n_function_words_diag": fw,
        "function_word_ratio_diag": round(fw / n, 4) if n else 0,
        "short_token_ratio_diag": round(short / n, 4) if n else 0,
    })

token_quality = pd.DataFrame(fw_ratios)
save_df(token_quality.describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).T.reset_index().rename(columns={"index": "metrica"}), "06_tokens_metricas_resumo.csv")
display(token_quality.describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99]).T)


Fonte dos tokens para diagnóstico: tokenizacao_diagnostica_a_partir_de:texto_analise
Salvo: data/diagnostics/06_tokens_top5000.csv | shape=(5000, 5)


,token,freq,pct_sobre_total_tokens,is_function_word,is_short_le_2
0,que,99008,0.050914,True,False
1,speaker,79068,0.040660,False,False
2,eu,48637,0.025011,True,True
3,de,45687,0.023494,True,True
4,nao,30670,0.015772,True,False
5,meeting,24775,0.012740,False,False
6,voce,24542,0.012621,True,False
7,tem,17031,0.008758,False,False
8,uma,16888,0.008685,True,False
9,como,16331,0.008398,True,False


Salvo: data/diagnostics/06_tokens_funcionais_top1000.csv | shape=(120, 5)


,token,freq,pct_sobre_total_tokens,is_function_word,is_short_le_2
0,que,99008,0.050914,True,False
2,eu,48637,0.025011,True,True
3,de,45687,0.023494,True,True
4,nao,30670,0.015772,True,False
6,voce,24542,0.012621,True,False
8,uma,16888,0.008685,True,False
9,como,16331,0.008398,True,False
11,para,16292,0.008378,True,False
12,no,16037,0.008247,True,True
14,mas,15748,0.008098,True,False


Salvo: data/diagnostics/06_tokens_metricas_resumo.csv | shape=(6, 15)


,count,mean,std,min,1%,5%,10%,25%,50%,75%,90%,95%,99%,max
n_tokens_diag,12041.0,161.498131,22.466772,17.0000,103.40000,124.0000,133.0000,148.0000,165.0000,177.0000,187.0000,192.0000,200.00000,222.0000
n_tokens_unicos_diag,12041.0,95.319160,15.306428,15.0000,50.00000,68.0000,75.0000,87.0000,98.0000,106.0000,112.0000,116.0000,122.00000,131.0000
token_unique_ratio_diag,12041.0,0.591046,0.057065,0.1338,0.45360,0.5000,0.5234,0.5565,0.5917,0.6266,0.6582,0.6779,0.72706,0.9474
n_function_words_diag,12041.0,53.975334,14.775124,2.0000,18.00000,29.0000,34.0000,44.0000,55.0000,64.0000,72.0000,77.0000,86.00000,104.0000
function_word_ratio_diag,12041.0,0.328418,0.058592,0.0625,0.17372,0.2266,0.2521,0.2929,0.3333,0.3687,0.4000,0.4167,0.44878,0.5692
short_token_ratio_diag,12041.0,0.195145,0.047480,0.0636,0.10000,0.1271,0.1391,0.1620,0.1900,0.2229,0.2597,0.2826,0.32396,0.5000


## 11. Ruídos de transcrição e PII residual


In [11]:
NOISE_PATTERNS = {
    "audio_video_filename": r"\b[\w\-.]+\.(?:mp3|m4a|wav|aac|ogg|flac|mp4|mov)\b",
    "generic_filename": r"\b[\w\-.]+\.(?:txt|pdf|docx?|xlsx?|csv|json)\b",
    "timestamp_hhmmss": r"\b\d{1,2}:\d{2}(?::\d{2})?\b",
    "speaker_label_long": r"\b(?:speaker|entrevistador[a]?|pesquisador[a]?|respondente|participante|moderador[a]?)\s*\d*\s*:",
    "speaker_label_short": r"\b[A-Z]{1,4}\d*\s*:",
    "date_like": r"\b(?:\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}|\d{4}[/\-]\d{1,2}[/\-]\d{1,2})\b",
    "email": r"\b[\w\.-]+@[\w\.-]+\.\w+\b",
    "phone_br_like": r"\b(?:\+?55\s?)?\(?\d{2}\)?\s?9?\d{4}[-\s]?\d{4}\b",
    "cpf_like": r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
    "cep_like": r"\b\d{5}-?\d{3}\b",
    "url": r"https?://\S+|www\.\S+",
}

noise_rows = []
texts = df_tokens[text_col_tok].fillna("").astype(str)
for name, pattern in NOISE_PATTERNS.items():
    counts = texts.str.count(pattern, flags=re.IGNORECASE)
    noise_rows.append({
        "flag": name,
        "chunks_com_ocorrencia": int((counts > 0).sum()),
        "pct_chunks": round(float((counts > 0).mean()), 4),
        "total_ocorrencias": int(counts.sum()),
    })
noise_summary = pd.DataFrame(noise_rows).sort_values(["chunks_com_ocorrencia", "total_ocorrencias"], ascending=False)
save_df(noise_summary, "07_ruidos_pii_resumo.csv")
display(noise_summary)

# Amostras redigidas, evitando expor PII no CSV.
def redact_noise(text: str) -> str:
    out = str(text)
    replacements = {
        "email": "[EMAIL]",
        "phone_br_like": "[TELEFONE]",
        "cpf_like": "[CPF]",
        "cep_like": "[CEP]",
        "url": "[URL]",
    }
    for key, repl in replacements.items():
        out = re.sub(NOISE_PATTERNS[key], repl, out, flags=re.IGNORECASE)
    return out[:1000]

sample_indices = set()
for name, pattern in NOISE_PATTERNS.items():
    idxs = texts[texts.str.contains(pattern, flags=re.IGNORECASE, regex=True, na=False)].index[:50]
    sample_indices.update(idxs)

noise_sample_cols = [c for c in [doc_col, chunk_col, idx_col, cols_tokens.get("language"), text_col_tok] if c and c in df_tokens.columns]
noise_sample = df_tokens.loc[sorted(sample_indices), noise_sample_cols].head(1000).copy()
if len(noise_sample):
    noise_sample["texto_redigido"] = noise_sample[text_col_tok].apply(redact_noise)
    noise_sample = noise_sample.drop(columns=[text_col_tok])
save_df(noise_sample, "07_ruidos_pii_amostra_texto_redigido.csv")
display(noise_sample.head(20))


Salvo: data/diagnostics/07_ruidos_pii_resumo.csv | shape=(11, 4)


,flag,chunks_com_ocorrencia,pct_chunks,total_ocorrencias
2,timestamp_hhmmss,12039,0.9998,91634
4,speaker_label_short,9929,0.8246,12473
0,audio_video_filename,8561,0.7110,10742
1,generic_filename,0,0.0000,0
3,speaker_label_long,0,0.0000,0
5,date_like,0,0.0000,0
6,email,0,0.0000,0
7,phone_br_like,0,0.0000,0
8,cpf_like,0,0.0000,0
9,cep_like,0,0.0000,0


Salvo: data/diagnostics/07_ruidos_pii_amostra_texto_redigido.csv | shape=(70, 5)


,doc_id,chunk_id,chunk_index,idioma,texto_redigido
0,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0001,1,pt,"Speaker 1 - 00:08\n \n Só minutinho, então eu vou esperar. \n \n \n \n \n \nSpeaker 2 - 00:09\n \n Os participantes que assistem, inclusive a Ana, entrarem aqui. Quando eu ligo a webinária, eles c..."
1,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0002,2,pt,"s. Antes de começar, aproveite para pegar uma água e ir ao banheiro. Assim, você não precisa \nsair durante a conversa. Agora, para que você não tenha nenhuma preocupação durante o nosso bate-papo..."
2,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0003,3,pt,"não grave nem tire prints ou fotos da tela durante o bate-papo. Agora falta muito pouco para \ncomeçar. Não se esqueça que estamos aqui para te ouvir, então não deixe de expressar suas opiniões ve..."
3,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0004,4,pt,"o projeto com vocês, então só reforçando que aqui não \ntem certo e errado, então tudo que vocês tiverem pra falar é legal pra gente ouvir, é pra ser uma reunião gostosa, \nde bate-papo, de troca ..."
4,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0005,5,pt,"tar, pra gente começar, quebrar o gelo. \n \n \n \n \n \nSpeaker 6 - 05:04\n \n Pode ser eu? \n \n \n \n \n \nSpeaker 2 - 05:05\n \n Vamos lá. \n \n \n \n \n \nSpeaker 4 - 05:06\n \n Tudo bom, vai..."
5,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0006,6,pt,"nho [IDADE] anos, sou \nestudante universitário, não tenho filhos, não sou solteiro e acredito que é isso. \n \n \n \n \nSpeaker 4 - 06:15\n \nMeeting Title: Q1_Biome_1104_9h00.m4a Meeting created..."
6,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0007,7,pt,"2 - 07:05\n \n Eu peguei esse monstro aqui do meu marido, que eu não achei o fone normal. Eu trabalho no ramo decoração de \ninteriores. A gente tem uma empresa, as versianas, janelas, né? A gente..."
7,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0008,8,pt,"m bacana, cada de lugar bacana. Obrigada, Isa. \nE você, Julinha? Vamos lá, querida. Chegou a sua vez. \n \n \n \n \nSpeaker 9 - 08:18\n \n Bom dia, eu sou a [NOME]. eu sou [NOME], do sul de Minas..."
8,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0009,9,pt,"aker 4 - 09:12\n \n No lugar, tem gente casada, tem gente que não é casada, tem gente com filhos ou não, enfim, tá muito variado, \nmas é legal. \n \n \n \n \n \nSpeaker 2 - 09:18\n \n Ouvir pouqu..."
9,doc_b53730e47cc798ef,doc_b53730e47cc798ef_ch_0010,10,pt,", trabalho pra casa. Eu gosto de sair com a família, sair com as \ncrianças. Meu hobby também é jogar futebol, nas horas vazias eu uso isso. Meu dia é bastante corrido, né? \nSempre buscando a mel..."


## 12. Qualidade dos chunks para modelagem

Esta seção prioriza problemas que costumam gerar clusters ruins: chunk curto demais, chunk longo demais, texto repetitivo, excesso de stopwords/fillers, idioma misto, muitos ruídos técnicos.


In [12]:
quality = pd.DataFrame(index=df_tokens.index)
quality["n_chars"] = stats_tok["n_chars"]
quality["n_words"] = stats_tok["n_words"]
quality = pd.concat([quality, token_quality.reindex(df_tokens.index)], axis=1)
quality["idioma_detectado_light"] = lang_audit["idioma_detectado_light"].values

# Conta ruídos por chunk
for name, pattern in NOISE_PATTERNS.items():
    quality[f"flag_{name}"] = texts.str.contains(pattern, flags=re.IGNORECASE, regex=True, na=False)
quality["n_noise_flags"] = quality[[c for c in quality.columns if c.startswith("flag_")]].sum(axis=1)

quality["is_too_short"] = quality["n_words"] < 25
quality["is_too_long"] = quality["n_words"] > quality["n_words"].quantile(.99)
quality["is_low_lexical_diversity"] = quality["token_unique_ratio_diag"] < 0.25
quality["is_high_function_word_ratio"] = quality["function_word_ratio_diag"] > 0.45
quality["is_mixed_or_unknown_language"] = quality["idioma_detectado_light"].isin(["mixed", "unknown"])
quality["has_many_noise_flags"] = quality["n_noise_flags"] >= 2

problem_cols = [
    "is_too_short", "is_too_long", "is_low_lexical_diversity", "is_high_function_word_ratio",
    "is_mixed_or_unknown_language", "has_many_noise_flags",
]
quality["n_problem_flags"] = quality[problem_cols].sum(axis=1)
quality["quality_bucket"] = pd.cut(
    quality["n_problem_flags"],
    bins=[-1, 0, 1, 2, 99],
    labels=["ok", "atenção", "problemático", "crítico"]
)

quality_export_cols = [c for c in [doc_col, chunk_col, idx_col, cols_tokens.get("language"), text_col_tok] if c and c in df_tokens.columns]
quality_export = pd.concat([df_tokens[quality_export_cols], quality], axis=1)
save_df(quality_export, "08_qualidade_chunks_completa.csv")

flag_summary = pd.DataFrame({
    "flag": problem_cols,
    "n_chunks": [int(quality[c].sum()) for c in problem_cols],
    "pct_chunks": [round(float(quality[c].mean()), 4) for c in problem_cols],
}).sort_values("n_chunks", ascending=False)
save_df(flag_summary, "08_qualidade_flags_resumo.csv")
display(flag_summary)

top_review = quality_export.sort_values(["n_problem_flags", "n_noise_flags", "n_words"], ascending=[False, False, True]).head(1000)
save_df(top_review, "08_chunks_para_revisao_top1000.csv")
display(top_review.head(30))


Salvo: data/diagnostics/08_qualidade_chunks_completa.csv | shape=(12041, 34)
Salvo: data/diagnostics/08_qualidade_flags_resumo.csv | shape=(6, 3)


,flag,n_chunks,pct_chunks
5,has_many_noise_flags,9984,0.8292
4,is_mixed_or_unknown_language,3138,0.2606
1,is_too_long,120,0.0100
3,is_high_function_word_ratio,108,0.0090
0,is_too_short,11,0.0009
2,is_low_lexical_diversity,7,0.0006


Salvo: data/diagnostics/08_chunks_para_revisao_top1000.csv | shape=(1000, 34)


,doc_id,chunk_id,chunk_index,idioma,texto_analise,n_chars,n_words,n_tokens_diag,n_tokens_unicos_diag,token_unique_ratio_diag,n_function_words_diag,function_word_ratio_diag,short_token_ratio_diag,idioma_detectado_light,flag_audio_video_filename,flag_generic_filename,flag_timestamp_hhmmss,flag_speaker_label_long,flag_speaker_label_short,flag_date_like,flag_email,flag_phone_br_like,flag_cpf_like,flag_cep_like,flag_url,n_noise_flags,is_too_short,is_too_long,is_low_lexical_diversity,is_high_function_word_ratio,is_mixed_or_unknown_language,has_many_noise_flags,n_problem_flags,quality_bucket
6671,doc_474985d7ea9b5b8b,doc_474985d7ea9b5b8b_ch_0037,37,es,"44:06\n \n No, en lo absoluto, ya hay demasiado de eso para que algo que tiene que ver con el bienestar y con el poder estar \nbien se convierta en una obligación más y en algo que hay que decir, ...",1195,201,201,102,0.5075,93,0.4627,0.2587,mixed,True,False,True,False,True,False,False,False,False,False,False,3,False,True,False,True,True,True,4,crítico
12001,doc_948ae088c8e6f14c,doc_948ae088c8e6f14c_ch_0028,28,es,"mencionan o que está estructurado, es \ncomo que está en equilibrio con la naturaleza como comentaron hace rato, como decirlo así como \nMeeting Title: DG4_Jack Pearson_3110_22h00.m4a Meeting crea...",1198,202,202,98,0.4851,94,0.4653,0.2921,mixed,True,False,True,False,True,False,False,False,False,False,False,3,False,True,False,True,True,True,4,crítico
631,doc_96f34760018e32cc,doc_96f34760018e32cc_ch_0075,75,es,"o podría. Sí, adelante. \n \n \n \n \n \nSpeaker 3 - 01:23:53\n \n ¿Es como que te interroga, como que te plantea algo y sí, o sea, si cambias esto, como que la marca en sí yo creo \nque plantea e...",1196,208,208,93,0.4471,99,0.4760,0.3317,mixed,False,False,True,False,True,False,False,False,False,False,False,2,False,True,False,True,True,True,4,crítico
9589,doc_047d9aa2d7261b26,doc_047d9aa2d7261b26_ch_0059,59,pt,"4h00.mp3 Meeting created at: 16th Mar, 2026 - 4:38 PM\n65 / 156\n Piorou. \n \n \n \n \n \nSpeaker 5 - 54:58\n \n Piorou. \n \n \n \n \nSpeaker 1 - 54:59\n \n Piorou. \n \n \n \n \n \nSpeaker 2 - ...",1196,89,89,22,0.2472,8,0.0899,0.1798,mixed,True,False,True,False,True,False,False,False,False,False,False,3,False,False,True,False,True,True,3,crítico
9988,doc_0dc2e6ecde931dc0,doc_0dc2e6ecde931dc0_ch_0117,117,pt,"ing Title: DG6_Mercato_221123_14h00.mp3 Meeting created at: 16th Mar, 2026 - 4:45 PM\n117 / 145\n \n \n \nSpeaker 5 - 01:44:33\n \n Na época errada. Tem que enxergar. Tem que enxergar. \n \n \n \n...",1193,115,115,18,0.1565,25,0.2174,0.1391,mixed,True,False,True,False,True,False,False,False,False,False,False,3,False,False,True,False,True,True,3,crítico
10110,doc_5e9eef5d22c00bcf,doc_5e9eef5d22c00bcf_ch_0099,99,pt,á antes? \n \n \n \n \n \nSpeaker 6 - 01:34:57\n \n Ela foi lá antes? Ela foi lá antes? \n \n \n \n \nSpeaker 1 - 01:35:01\n \n Ela foi lá antes? \n \n \n \n \nSpeaker 7 - 01:35:03\n \n Ela foi lá...,1195,130,130,44,0.3385,74,0.5692,0.2308,mixed,True,False,True,False,True,False,False,False,False,False,False,3,False,False,False,True,True,True,3,crítico
11812,doc_a57807230362f233,doc_a57807230362f233_ch_0049,49,es,"- 01:01:50\n \n Pues suena mucho innovadoras y de tecnología, ¿No? Y con experiencia, ¿No? \n \n \n \n \n \nSpeaker 4 - 01:01:57\n \n Bueno, al menos como con creatividad, ¿No? Avances, \n \n \n \...",1198,158,158,89,0.5633,72,0.4557,0.2595,mixed,True,False,True,False,True,False,False,False,False,False,False,3,False,False,False,True,True,True,3,crítico
11836,doc_a57807230362f233,doc_a57807230362f233_ch_0073,73,es,"¿Yo sí estoy de acuerdo con él. \n \n \n \n \nSpeaker 1 - 01:32:53\n \n Qué pasó por ejemplo con bioeficiente, esa tecnología con qué tipo de beneficio de propuestas? ¿Las que están \nleyendo acá ...",1191,171,171,85,0.4971,83,0.4854,0.2749,mixed,True,False,True,False,True,False,False,False,False,False,False,3,False,False,False,True,True,True,3,crítico
12022,doc_948ae088c8e6f14c,doc_948ae088c8e6f14c_ch_0049,49,es,"3 - 59:36\n

## 13. Comparação dos chunks agregados contra entrevistas completas

Se os IDs baterem, esta seção compara o tamanho do texto completo com a soma dos chunks da mesma entrevista. Isso ajuda a detectar perda de trechos, duplicação e segmentação inconsistente.


In [13]:
if doc_tok and doc_full and doc_tok in df_tokens.columns and doc_full in df_full.columns:
    chunks_agg = df_tokens.groupby(doc_tok, dropna=False).agg(
        n_chunks=(text_col_tok, "size"),
        chars_chunks=(text_col_tok, lambda s: int(s.fillna("").astype(str).str.len().sum())),
        words_chunks=(text_col_tok, lambda s: int(s.fillna("").astype(str).apply(lambda x: len(simple_word_tokens(x))).sum())),
    ).reset_index().rename(columns={doc_tok: "doc_id_join"})

    full_stats = df_full[[doc_full, text_col_full]].copy().rename(columns={doc_full: "doc_id_join"})
    full_stats["chars_full"] = full_stats[text_col_full].fillna("").astype(str).str.len()
    full_stats["words_full"] = full_stats[text_col_full].fillna("").astype(str).apply(lambda x: len(simple_word_tokens(x)))
    full_stats = full_stats.drop(columns=[text_col_full])

    recon = chunks_agg.merge(full_stats, on="doc_id_join", how="outer", indicator=True)
    recon["word_coverage_chunks_vs_full"] = np.where(recon["words_full"] > 0, recon["words_chunks"] / recon["words_full"], np.nan)
    recon["char_coverage_chunks_vs_full"] = np.where(recon["chars_full"] > 0, recon["chars_chunks"] / recon["chars_full"], np.nan)
    save_df(recon, "09_reconciliacao_chunks_vs_full.csv")
    display(recon.describe(include="all").T)

    suspicious_recon = recon[
        (recon["_merge"] != "both")
        | (recon["word_coverage_chunks_vs_full"] < 0.70)
        | (recon["word_coverage_chunks_vs_full"] > 1.30)
    ].sort_values(["_merge", "word_coverage_chunks_vs_full"], na_position="first")
    save_df(suspicious_recon.head(1000), "09_reconciliacao_suspeitos_top1000.csv")
    display(suspicious_recon.head(30))
else:
    print("Sem IDs compatíveis detectados para comparar chunks contra entrevistas completas.")


Salvo: data/diagnostics/09_reconciliacao_chunks_vs_full.csv | shape=(155, 9)


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
doc_id_join,155,155,doc_012fe174842b390e,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
n_chunks,155.0,NaN,NaN,NaN,77.683871,36.63292,2.0,50.5,76.0,109.5,156.0
chars_chunks,155.0,NaN,NaN,NaN,92277.677419,43712.221341,1355.0,59503.0,89873.0,130487.0,185620.0
words_chunks,155.0,NaN,NaN,NaN,12545.8,5586.019143,164.0,8429.5,12514.0,17592.5,21793.0
chars_full,155.0,NaN,NaN,NaN,80953.335484,38314.177836,1206.0,52177.5,78768.0,114498.5,162745.0
words_full,155.0,NaN,NaN,NaN,10980.16129,4882.976273,146.0,7392.0,10966.0,15449.5,19035.0
_merge,155,1,both,155,NaN,NaN,NaN,NaN,NaN,NaN,NaN
word_coverage_chunks_vs_full,155.0,NaN,NaN,NaN,1.141678,0.00503,1.112125,1.140457,1.142405,1.144218,1.153292
char_coverage_chunks_vs_full,155.0,NaN,NaN,NaN,1.139074,0.004086,1.09593,1.13916,1.139854,1.140371,1.141301


Salvo: data/diagnostics/09_reconciliacao_suspeitos_top1000.csv | shape=(0, 9)


,doc_id_join,n_chunks,chars_chunks,words_chunks,chars_full,words_full,_merge,word_coverage_chunks_vs_full,char_coverage_chunks_vs_full


## 14. Diagnóstico rápido de clusterabilidade sem definir pipeline final

Esse bloco é apenas um sanity check. Ele ajuda a ver se o vocabulário ainda está dominado por ruído. O modelo final deve ser decidido depois de olhar os outputs anteriores.


In [14]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity

    def prep_for_tfidf_diag(text: str) -> str:
        toks = simple_word_tokens(text)
        toks = [t for t in toks if t not in ALL_FUNCTION_WORDS and len(t) >= 3]
        return " ".join(toks)

    MAX_DOCS_TFIDF = 15000
    tfidf_base = df_tokens.copy()
    if len(tfidf_base) > MAX_DOCS_TFIDF:
        tfidf_base = tfidf_base.sample(MAX_DOCS_TFIDF, random_state=42)

    corpus = tfidf_base[text_col_tok].fillna("").astype(str).apply(prep_for_tfidf_diag)
    vectorizer = TfidfVectorizer(min_df=3, max_df=0.60, ngram_range=(1, 2), max_features=8000)
    X = vectorizer.fit_transform(corpus)
    terms = np.array(vectorizer.get_feature_names_out())

    mean_tfidf = np.asarray(X.mean(axis=0)).ravel()
    top_terms = pd.DataFrame({"termo": terms, "mean_tfidf": mean_tfidf}).sort_values("mean_tfidf", ascending=False).head(1000)
    save_df(top_terms, "10_tfidf_termos_globais_top1000.csv")
    display(top_terms.head(80))

    sample_n = min(1500, X.shape[0])
    rng = np.random.default_rng(42)
    sample_positions = rng.choice(X.shape[0], size=sample_n, replace=False) if X.shape[0] > sample_n else np.arange(X.shape[0])
    sim = cosine_similarity(X[sample_positions])
    tri = sim[np.triu_indices_from(sim, k=1)]
    sim_summary = pd.DataFrame({
        "metrica": ["n_amostra", "similaridade_media", "similaridade_p50", "similaridade_p90", "similaridade_p95", "similaridade_p99"],
        "valor": [sample_n, float(np.mean(tri)), float(np.quantile(tri, .5)), float(np.quantile(tri, .9)), float(np.quantile(tri, .95)), float(np.quantile(tri, .99))]
    })
    save_df(sim_summary, "10_tfidf_similaridade_amostral_resumo.csv")
    display(sim_summary)
except Exception as e:
    print("Sanity check TF-IDF não rodou:", repr(e))


Salvo: data/diagnostics/10_tfidf_termos_globais_top1000.csv | shape=(1000, 2)


,termo,mean_tfidf
3192,gente,0.038358
7090,tem,0.037340
3996,mais,0.036663
5487,pra,0.030504
4501,muito,0.027537
593,assim,0.027472
4599,natura,0.026119
7600,vai,0.025396
71,acho,0.024931
6992,tambem,0.024269


Salvo: data/diagnostics/10_tfidf_similaridade_amostral_resumo.csv | shape=(6, 2)


,metrica,valor
0,n_amostra,1500.000000
1,similaridade_media,0.035193
2,similaridade_p50,0.028747
3,similaridade_p90,0.073918
4,similaridade_p95,0.093185
5,similaridade_p99,0.146438


## 15. Gráficos simples para inspeção rápida


In [15]:
if plt is None:
    print("matplotlib não disponível; pulando gráficos.")
else:
    # Um gráfico por arquivo para evitar confusão visual.
    plot_specs = [
        (stats_tok["n_words"], "Distribuição de palavras por chunk", "n_words_chunks.png"),
        (quality["function_word_ratio_diag"], "Razão de palavras funcionais por chunk", "function_word_ratio_chunks.png"),
        (quality["token_unique_ratio_diag"], "Diversidade lexical por chunk", "token_unique_ratio_chunks.png"),
    ]
    for series, title, filename in plot_specs:
        fig = plt.figure(figsize=(10, 5))
        series.dropna().clip(upper=series.dropna().quantile(.99)).hist(bins=50)
        plt.title(title)
        plt.xlabel(series.name or "valor")
        plt.ylabel("frequência")
        out = PLOTS_DIR / filename
        fig.savefig(out, bbox_inches="tight", dpi=140)
        plt.close(fig)
        print("Gráfico salvo:", out.relative_to(PROJECT_ROOT))


Gráfico salvo: data/diagnostics/plots/n_words_chunks.png
Gráfico salvo: data/diagnostics/plots/function_word_ratio_chunks.png
Gráfico salvo: data/diagnostics/plots/token_unique_ratio_chunks.png


## 16. Relatório final da execução


In [16]:
def df_to_md(df, max_rows=20):
    if df is None or len(df) == 0:
        return "_Sem dados._"
    try:
        return df.head(max_rows).to_markdown(index=False)
    except Exception:
        return df.head(max_rows).to_string(index=False)

lang_dist_md = df_to_md(dist_lang)
quality_md = df_to_md(quality["quality_bucket"].value_counts(dropna=False).reset_index().rename(columns={"index": "bucket", "quality_bucket": "n_chunks"}))
flags_md = df_to_md(flag_summary)
coverage_md = df_to_md(coverage)
noise_md = df_to_md(noise_summary)
source_tokens = BQ_TOKENS_TABLE_RESOLVED
source_full = BQ_FULL_TABLE_RESOLVED

report = f"""
# Diagnóstico da base de entrevistas

Run ID: `{RUN_ID}`  
Gerado em: `{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}`

## Fontes BigQuery

- Base segmentada/chunks/tokens: `{source_tokens}`
- Base de entrevistas completas: `{source_full}`

## Tamanho das bases

| base | linhas | colunas |
|---|---:|---:|
| tokens/chunks | {df_tokens.shape[0]:,} | {df_tokens.shape[1]:,} |
| entrevistas completas | {df_full.shape[0]:,} | {df_full.shape[1]:,} |

## Colunas detectadas

### Tokens/chunks
```json
{json.dumps(cols_tokens, ensure_ascii=False, indent=2)}
```

### Entrevistas completas
```json
{json.dumps(cols_full, ensure_ascii=False, indent=2)}
```

## Cobertura e IDs

{coverage_md}

## Distribuição de idioma detectado

{lang_dist_md}

## Qualidade dos chunks

{quality_md}

## Flags de qualidade

{flags_md}

## Ruídos/PII detectados

{noise_md}

## Arquivos principais para olhar antes do próximo notebook

1. `01_colunas_detectadas.csv` — confirma se as colunas certas foram inferidas.
2. `04_segmentacao_por_documento.csv` — mostra gaps, duplicidade de índice e tamanho dos chunks por entrevista.
3. `04_overlap_chunks_consecutivos.csv` — mostra repetição/overlap entre chunks consecutivos.
4. `05_idioma_mismatch_amostra.csv` e `05_idioma_misto_amostra.csv` — apontam mistura de idioma.
5. `06_tokens_funcionais_top1000.csv` — identifica preposições/fillers que estavam dominando clusters.
6. `07_ruidos_pii_resumo.csv` — mostra ruídos de transcrição e PII residual.
7. `08_chunks_para_revisao_top1000.csv` — prioriza chunks problemáticos.
8. `09_reconciliacao_chunks_vs_full.csv` — compara chunks agregados contra entrevistas completas.
9. `10_tfidf_termos_globais_top1000.csv` — sanity check de vocabulário pós-limpeza diagnóstica.

## Interpretação esperada

Este notebook deve responder:

- A tabela completa realmente foi baixada e bate com a tabela de chunks?
- Os IDs estão consistentes?
- A segmentação atual tem gaps, duplicatas ou overlap excessivo?
- Há mistura de idiomas relevante?
- O vocabulário está dominado por stopwords, fillers ou ruídos técnicos?
- Quais chunks devem ser revisados antes de definir a limpeza final?
""".strip()

save_text(report, "diagnostico_base_resumo.md")
print(report[:4000])


Salvo: data/diagnostics/diagnostico_base_resumo.md
# Diagnóstico da base de entrevistas

Run ID: `20260519_234444`  
Gerado em: `2026-05-19 23:45:32`

## Fontes BigQuery

- Base segmentada/chunks/tokens: `kyra-pml-20261.kyra_base_2026_1.entrevistas_chunks_v2`
- Base de entrevistas completas: `kyra-pml-20261.kyra_base_2026_1.entrevistas_docs_v2`

## Tamanho das bases

| base | linhas | colunas |
|---|---:|---:|
| tokens/chunks | 12,041 | 32 |
| entrevistas completas | 155 | 29 |

## Colunas detectadas

### Tokens/chunks
```json
{
  "doc_id": "doc_id",
  "chunk_id": "chunk_id",
  "chunk_index": "chunk_index",
  "language": "idioma",
  "text": "texto_analise",
  "tokens": "n_palavras",
  "date": "timestamp_inicial"
}
```

### Entrevistas completas
```json
{
  "doc_id": "doc_id",
  "chunk_id": null,
  "chunk_index": null,
  "language": "idioma",
  "text": "texto_analise",
  "tokens": "n_palavras_total",
  "date": "data_bruta_nome_arquivo"
}
```

## Cobertura e IDs

| base                  

## 17. Checklist antes do próximo notebook de pré-processamento

Depois de rodar tudo, abra a pasta `data/diagnostics` e valide estes pontos:

1. `00_bq_catalogo_tabelas.csv`: confirma que as duas tabelas do BigQuery foram encontradas.
2. `01_colunas_detectadas.csv`: confirme se `doc_id`, `chunk_id`, `chunk_index`, `idioma` e texto foram detectados corretamente.
3. `04_segmentacao_por_documento.csv`: procure documentos com gaps ou índices duplicados.
4. `04_overlap_alto_chunks_consecutivos.csv`: se houver muitos casos, a segmentação precisa ser revista.
5. `05_idioma_mismatch_amostra.csv`: valide se existe erro real de idioma ou se é ruído do detector simples.
6. `06_tokens_funcionais_top1000.csv`: use para montar stopwords controladas, sem remover negações importantes.
7. `07_ruidos_pii_resumo.csv`: decida padrões de limpeza antes da tokenização definitiva.
8. `09_reconciliacao_chunks_vs_full.csv`: se a cobertura for muito baixa ou alta demais, a base de chunks não pode ser usada sem reconstrução.
